# Fine-tune GPT-2 Vietnamese for Math Word Problems

**Goal:** fine-tune `NlpHUST/gpt2-vietnamese` to solve Vietnamese math word problems.

**Inputs:** `train.json`, `valid.json`, and the local GPT-2 model folder from Kaggle Input.

**Pipeline:** load data → SFT training → generate validation outputs → evaluate by relative error.

**Rules:** Internet OFF, no extra data/API/LLM, total runtime ≤ 3 hours.


In [1]:
import json
import re
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple
import time
import datetime

In [2]:
# ============================================================
# 1. Analysis and Sample Filterings
# ============================================================

# ============================================================
# PART 1: LOAD DATA
# ============================================================
 
def load_records(path: str | Path) -> list:
    """Load JSON file (supports both array and JSONL format)."""
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        # Check if starts with '[' (array) or '{' (JSONL)
        if head == "[":
            return json.load(f)
        else:
            # JSONL format
            return [json.loads(line) for line in f if line.strip()]
 
 
def save_records(records: list, path: str | Path):
    """Save records to JSON file."""
    p = Path(path)
    with p.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved {len(records)} records to {p}")

In [3]:
# ============================================================
# PART 2: ANALYZE DATA QUALITY
# ============================================================
 
def analyze_dataset(records: list[dict]) -> dict:
    """
    Comprehensive analysis of dataset quality.
    
    Returns:
        dict with statistics about the dataset
    """
    print(f"\n{'='*70}")
    print(f"ANALYZING {len(records)} RECORDS")
    print(f"{'='*70}\n")
    
    stats = {
        "total_records": len(records),
        "missing_fields": 0,
        "missing_query": 0,
        "missing_response": 0,
        "duplicate_queries": 0,
        "query_length_stats": [],
        "response_length_stats": [],
        "type_distribution": Counter(),
        "issues_by_record": [],
    }
    
    seen_queries = {}
    
    for idx, rec in enumerate(records):
        query = rec.get("query_vi", "").strip()
        response = rec.get("response_vi", "").strip()
        rec_type = rec.get("type", "unknown")
        
        issues = []
        original_idx = None
        
        # ===== CHECK 1: Missing fields =====
        if not query:
            stats["missing_query"] += 1
            issues.append("missing_query")
        
        if not response:
            stats["missing_response"] += 1
            issues.append("missing_response")
        
        if not query or not response:
            stats["missing_fields"] += 1
            issues.append("missing_fields")
        
        # ===== CHECK 2: Duplicate queries =====
        if query and query in seen_queries:
            stats["duplicate_queries"] += 1
            issues.append("duplicate")
            original_idx = seen_queries[query]
        if query and query not in seen_queries:
            seen_queries[query] = idx
        
        # ===== CHECK 3: Record type =====
        stats["type_distribution"][rec_type] += 1
        
        # ===== CHECK 4: Lengths =====
        if query:
            stats["query_length_stats"].append(len(query))
        if response:
            stats["response_length_stats"].append(len(response))
        
        # Store issues for detailed inspection
        if issues:
            stats["issues_by_record"].append({
                "idx": idx,
                "original_idx": original_idx,
                "issues": issues,
                "type": rec_type,
                "query_len": len(query),
                "response_len": len(response),
            })
    
    return stats
 
 
def print_analysis_report(stats: dict):
    """Pretty-print analysis results."""
    
    print("\n" + "="*70)
    print("DATA QUALITY REPORT")
    print("="*70)
    
    # Summary
    print(f"\nTotal records: {stats['total_records']}")
    print(f"Missing fields: {stats['missing_fields']} ({stats['missing_fields']/stats['total_records']*100:.1f}%)")
    print(f"  - Missing query: {stats['missing_query']}")
    print(f"  - Missing response: {stats['missing_response']}")
    print(f"Duplicate queries: {stats['duplicate_queries']} ({stats['duplicate_queries']/stats['total_records']*100:.1f}%)")
    
    # Type distribution
    print(f"\nType distribution:")
    for typ, count in stats["type_distribution"].most_common():
        pct = count / stats['total_records'] * 100
        print(f"  {typ:20s}: {count:6d} ({pct:5.1f}%)")
    
    # Length statistics
    if stats["query_length_stats"]:
        ql = sorted(stats["query_length_stats"])
        print(f"\nQuery length statistics:")
        print(f"  Min:     {min(ql):6d} chars")
        print(f"  Q1 (25%): {ql[len(ql)//4]:6d} chars")
        print(f"  Median:  {ql[len(ql)//2]:6d} chars")
        print(f"  Q3 (75%): {ql[3*len(ql)//4]:6d} chars")
        print(f"  Max:     {max(ql):6d} chars")
        print(f"  Mean:    {sum(ql)/len(ql):6.0f} chars")
    
    if stats["response_length_stats"]:
        rl = sorted(stats["response_length_stats"])
        print(f"\nResponse length statistics:")
        print(f"  Min:     {min(rl):6d} chars")
        print(f"  Q1 (25%): {ql[len(rl)//4]:6d} chars")
        print(f"  Median:  {rl[len(rl)//2]:6d} chars")
        print(f"  Q3 (75%): {rl[3*len(rl)//4]:6d} chars")
        print(f"  Max:     {max(rl):6d} chars")
        print(f"  Mean:    {sum(rl)/len(rl):6.0f} chars")
    
    # Issues
    if stats["issues_by_record"]:
        print(f"\n⚠️  RECORDS WITH ISSUES: {len(stats['issues_by_record'])}")
        dup_with_original = sum(1 for r in stats["issues_by_record"] if "duplicate" in r["issues"] and r.get("original_idx") is not None)
        if dup_with_original > 0:
            print(f"  👉 Trong đó có {dup_with_original} câu trùng lặp đã xác định được vị trí câu gốc.")
        print(f"\nTop 5 issues:")
        issue_counts = Counter()
        for r in stats["issues_by_record"]:
            for issue in r["issues"]:
                issue_counts[issue] += 1
        for issue, count in issue_counts.most_common(5):
            pct = count / stats['total_records'] * 100
            print(f"  {issue:20s}: {count:6d} ({pct:5.1f}%)")
    
    print("\n" + "="*70)

In [4]:
# ============================================================
# PART 3: INSPECT PROBLEMATIC RECORDS
# ============================================================
 
def show_problematic_records(records: list[dict], stats: dict, limit: int = 5):
    """Show examples of records with issues."""
    
    if not stats["issues_by_record"]:
        print("✓ No problematic records found!")
        return
    
    print(f"\n{'='*70}")
    print(f"EXAMPLES OF PROBLEMATIC RECORDS (showing {limit})")
    print(f"{'='*70}\n")
    
    for issue_rec in stats["issues_by_record"][:limit]:
        idx = issue_rec["idx"]
        rec = records[idx]
        
        print(f"Index: {idx}")
        print(f"Type: {issue_rec['type']}")
        print(f"Issues: {', '.join(issue_rec['issues'])}")
        if "duplicate" in issue_rec["issues"] and issue_rec["original_idx"] is not None:
            orig_idx = issue_rec["original_idx"]
            print(f"👉 CÂU SONG SINH GỐC NẰM TẠI INDEX: {orig_idx}")
            print(f"   [Lời giải câu gốc]: {records[orig_idx].get('response_vi', '')}")
            print(f"   ---------------------------------------------")
        print(f"Lengths: query={issue_rec['query_len']}, response={issue_rec['response_len']}")
        print(f"Query: {rec.get('query_vi', '')}")
        print(f"Response: {rec.get('response_vi', '')}")
        print("-" * 70 + "\n")

In [5]:
# ============================================================
# PART 4: FILTER & CLEAN DATA
# ============================================================
 
def filter_and_clean(
    records: list[dict],
    min_query_len: int = 20,
    max_query_len: int = 500,
    min_response_len: int = 30,
    max_response_len: int = 1000,
    require_answer_anchor: bool = True,
    remove_duplicates: bool = True,
    verbose: bool = True,
) -> list[dict]:
    """
    Filter and clean records based on multiple criteria.
    
    Args:
        records: List of records to filter
        min_query_len: Minimum query length (default 20)
        max_query_len: Maximum query length (default 500)
        min_response_len: Minimum response length (default 30)
        max_response_len: Maximum response length (default 1000)
        require_answer_anchor: Require answer marker (####, Đáp án là, etc.)
        remove_duplicates: Remove duplicate queries
        verbose: Print detailed report
    
    Returns:
        Cleaned list of records
    """
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"FILTERING DATA")
        print(f"{'='*70}\n")
        print(f"Criteria:")
        print(f"  Query length: {min_query_len}-{max_query_len} chars")
        print(f"  Response length: {min_response_len}-{max_response_len} chars")
        print(f"  Require answer anchor: {require_answer_anchor}")
        print(f"  Remove duplicates: {remove_duplicates}\n")
    
    # Answer anchors to look for
    ANCHORS = ["####", "Đáp án là", "Câu trả lời là", "Đáp án", "đáp án là", "câu trả lời là"]
    
    cleaned = []
    removed_reasons = {
        "missing_fields": 0,
        "query_too_short": 0,
        "query_too_long": 0,
        "response_too_short": 0,
        "response_too_long": 0,
        "no_anchor": 0,
        "duplicate": 0,
    }
    
    seen_queries = set()
    
    for rec in records:
        q = rec.get("query_vi", "").strip()
        r = rec.get("response_vi", "").strip()
        
        # ===== CHECK 1: Missing fields =====
        if not q or not r:
            removed_reasons["missing_fields"] += 1
            continue
        
        # ===== CHECK 2: Duplicate queries =====
        if remove_duplicates and q in seen_queries:
            removed_reasons["duplicate"] += 1
            continue
        if remove_duplicates:
            seen_queries.add(q)
        
        # ===== CHECK 3: Query length =====
        if len(q) < min_query_len:
            removed_reasons["query_too_short"] += 1
            continue
        
        if len(q) > max_query_len:
            removed_reasons["query_too_long"] += 1
            continue
        
        # ===== CHECK 4: Response length =====
        if len(r) < min_response_len:
            removed_reasons["response_too_short"] += 1
            continue
        
        if len(r) > max_response_len:
            removed_reasons["response_too_long"] += 1
            continue
        
        # ===== CHECK 5: Answer anchor =====
        if require_answer_anchor and not any(anc in r for anc in ANCHORS):
            removed_reasons["no_anchor"] += 1
            continue
        
        # ===== PASSED ALL CHECKS =====
        cleaned.append({
            **rec,
            "query_vi": q,
            "response_vi": r,
        })
    
    # Print report
    if verbose:
        print(f"Original records:  {len(records):6d}")
        print(f"Cleaned records:   {len(cleaned):6d}")
        print(f"Removed:           {len(records) - len(cleaned):6d} ({(len(records)-len(cleaned))/len(records)*100:.1f}%)\n")
        
        print(f"Removed by reason:")
        for reason, count in removed_reasons.items():
            if count > 0:
                pct = count / len(records) * 100
                print(f"  {reason:25s}: {count:6d} ({pct:5.1f}%)")
    
    return cleaned

In [6]:
# ============================================================
# PART 5: SAMPLE & INSPECT CLEANED DATA
# ============================================================
 
def sample_records(records: list[dict], sample_size: int = 10, seed: int = 41) -> list[dict]:
    """Random sample from records."""
    import random
    random.seed(seed)
    return random.sample(records, min(sample_size, len(records)))
 
 
def show_samples(records: list[dict], sample_size: int = 5, title: str = "Sample Records"):
    """Display sample records."""
    samples = sample_records(records, sample_size)
    
    print(f"\n{'='*70}")
    print(f"{title}")
    print(f"{'='*70}\n")
    
    for i, rec in enumerate(samples, 1):
        print(f"Sample {i}:")
        print(f"Type: {rec.get('type', 'N/A')}")
        print(f"Query ({len(rec.get('query_vi', ''))} chars):")
        print(f"  {rec.get('query_vi', '')}...")
        print(f"\nResponse ({len(rec.get('response_vi', ''))} chars):")
        print(f"  {rec.get('response_vi', '')}...")
        print("-" * 70 + "\n")

In [7]:
# ============================================================
# PART 6: CHECK FOR SPECIFIC ISSUES
# ============================================================

def check_answer_anchors(records: list[dict]) -> dict:
    """Check which answer anchors are used in responses."""
    anchors_found = Counter()
    records_without_anchor = []
    
    ANCHORS = ["####", "Đáp án là", "Câu trả lời là", "Đáp án", "câu trả lời là", "đáp án là", "The answer is"]
    
    for idx, rec in enumerate(records):
        r = rec.get("response_vi", "")
        
        found_any = False
        for anc in ANCHORS:
            if anc in r:
                anchors_found[anc] += 1
                found_any = True
        
        if not found_any:
            records_without_anchor.append((idx, rec))
    
    print(f"\n{'='*70}")
    print(f"ANSWER ANCHOR ANALYSIS")
    print(f"{'='*70}\n")
    
    print(f"Anchors used:")
    for anchor, count in anchors_found.most_common():
        pct = count / len(records) * 100
        print(f"  '{anchor:15s}': {count:6d} ({pct:5.1f}%)")
    
    print(f"\nRecords WITHOUT answer anchor: {len(records_without_anchor)} ({len(records_without_anchor)/len(records)*100:.1f}%)")
    
    if records_without_anchor:
        print(f"\nExamples (first 10):")
        for idx, rec in records_without_anchor[:10]:
            print(f"  Index {idx}:")
            print(f"    Response: {rec.get('response_vi', '')}")
    
    return {
        "anchors_found": anchors_found,
        "without_anchor": records_without_anchor,
    }
 
 
def check_for_repetition(records: list[dict], min_repeat_len: int = 20) -> dict:
    """Check for obviously repeated content in responses."""
    repetitive = []
    
    for idx, rec in enumerate(records):
        r = rec.get("response_vi", "")
        lines = r.split('\n')
        
        # Check for duplicate lines
        seen_lines = set()
        duplicates = []
        for line in lines:
            line_clean = line.strip()
            if len(line_clean) > min_repeat_len:
                if line_clean in seen_lines:
                    duplicates.append(line_clean)
                seen_lines.add(line_clean)
        
        if duplicates:
            repetitive.append({
                "idx": idx,
                "type": rec.get("type"),
                "num_duplicates": len(duplicates),
                "examples": duplicates[:2],
            })
    
    print(f"\n{'='*70}")
    print(f"REPETITION ANALYSIS")
    print(f"{'='*70}\n")
    
    print(f"Records with repeated lines: {len(repetitive)} ({len(repetitive)/len(records)*100:.1f}%)")
    
    if repetitive:
        print(f"\nExamples (first 3):")
        for rep in repetitive[:3]:
            print(f"  Index {rep['idx']} (type: {rep['type']}):")
            print(f"    Duplicate lines: {rep['num_duplicates']}")
            for ex in rep['examples']:
                print(f"      '{ex[:80]}'")
    
    return {
        "repetitive_records": repetitive,
    }
 
 
def check_numeric_content(records: list[dict]) -> dict:
    """Check if responses contain numeric answers."""
    with_numbers = 0
    without_numbers = 0
    
    for rec in records:
        r = rec.get("response_vi", "")
        if re.search(r'\d+', r):
            with_numbers += 1
        else:
            without_numbers += 1
    
    print(f"\n{'='*70}")
    print(f"NUMERIC CONTENT ANALYSIS")
    print(f"{'='*70}\n")
    
    print(f"Responses with numbers: {with_numbers} ({with_numbers/len(records)*100:.1f}%)")
    print(f"Responses without numbers: {without_numbers} ({without_numbers/len(records)*100:.1f}%)")
    
    return {
        "with_numbers": with_numbers,
        "without_numbers": without_numbers,
    }

In [8]:
# ============================================================
# PART 7: FULL PIPELINE
# ============================================================
 
def full_data_quality_pipeline(
    train_path: str | Path,
    valid_path: str | Path = None,
    output_dir: Path = None,
    filter_config: dict = None,
) -> Tuple[list[dict], list[dict], dict]:
    """
    Complete pipeline: load, analyze, filter, and save data.
    
    Args:
        train_path: Path to training data
        valid_path: Path to validation data (optional)
        output_dir: Where to save cleaned data
        filter_config: Custom filtering parameters
    
    Returns:
        (cleaned_train, cleaned_valid, analysis_results)
    """
    pipeline_start = time.time()
    
    if filter_config is None:
        filter_config = {
            "min_query_len": 20,
            "max_query_len": 500,
            "min_response_len": 30,
            "max_response_len": 1000,
            "require_answer_anchor": True,
            "remove_duplicates": True,
        }
    
    if output_dir is None:
        output_dir = Path("./cleaned_data")
    
    output_dir.mkdir(exist_ok=True)
    
    print("\n" + "🔍 " * 20)
    print("DATA QUALITY PIPELINE")
    print("🔍 " * 20)
    
    # ===== LOAD =====
    print(f"\n1️⃣  LOADING DATA")
    train_records = load_records(train_path)
    print(f"   ✓ Loaded {len(train_records)} training records")
    
    valid_records = None
    if valid_path:
        valid_records = load_records(valid_path)
        print(f"   ✓ Loaded {len(valid_records)} validation records")
    
    # ===== ANALYZE TRAINING DATA =====
    print(f"\n2️⃣  ANALYZING TRAINING DATA")
    train_stats = analyze_dataset(train_records)
    print_analysis_report(train_stats)
    show_problematic_records(train_records, train_stats, limit=3)
    
    # ===== CHECK FOR SPECIFIC ISSUES =====
    print(f"\n3️⃣  CHECKING FOR SPECIFIC ISSUES")
    check_answer_anchors(train_records)
    check_for_repetition(train_records)
    check_numeric_content(train_records)
    
    # ===== FILTER TRAINING DATA =====
    print(f"\n4️⃣  FILTERING TRAINING DATA")
    train_cleaned = filter_and_clean(train_records, **filter_config)
    
    # ===== FILTER VALIDATION DATA =====
    valid_cleaned = None
    if valid_records:
        print(f"\n5️⃣  FILTERING VALIDATION DATA")
        valid_cleaned = filter_and_clean(valid_records, **filter_config)
    else:
        print(f"\n5️⃣  SKIPPING VALIDATION DATA (not provided)")
    
    # ===== SAVE CLEANED DATA =====
    print(f"\n6️⃣  SAVING CLEANED DATA")
    save_records(train_cleaned, output_dir / "train_cleaned.json")
    if valid_cleaned:
        save_records(valid_cleaned, output_dir / "valid_cleaned.json")

    pipeline_end = time.time()
    elapsed_seconds = pipeline_end - pipeline_start
    formatted_time = str(datetime.timedelta(seconds=int(elapsed_seconds)))
    
    # ===== SHOW SAMPLES =====
    print(f"\n7️⃣  SAMPLES FROM CLEANED DATA")
    show_samples(train_cleaned, sample_size=3, title="Training Samples (Cleaned)")
    
    print("\n" + "✅ " * 20)
    print("PIPELINE COMPLETE")
    print(f"⏱️ Tổng thời gian xử lý dữ liệu: {formatted_time} (hh:mm:ss)")
    print("✅ " * 20 + "\n")
    
    return train_cleaned, valid_cleaned, {
        "train_stats": train_stats,
    }

In [9]:
# ============================================================
# Sample Filterings Main Execution
# ============================================================
 
if __name__ == "__main__":
    
    
    # Replace these with your actual file paths
    TRAIN_FILE = "/kaggle/input/datasets/kimanh2002/dataset-math/train.json"
    VALID_FILE = "/kaggle/input/datasets/kimanh2002/dataset-math/valid.json"
    OUTPUT_DIR = Path("/kaggle/working/cleaned_data")
    
    # Run full pipeline
    train_clean, valid_clean, results = full_data_quality_pipeline(
        train_path=TRAIN_FILE,
        valid_path=VALID_FILE,
        output_dir=OUTPUT_DIR,
    )
    
    
    print("Data analysis module loaded successfully!")
    print("Use full_data_quality_pipeline() to run the complete pipeline")


🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 
DATA QUALITY PIPELINE
🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 🔍 

1️⃣  LOADING DATA
   ✓ Loaded 95400 training records
   ✓ Loaded 1000 validation records

2️⃣  ANALYZING TRAINING DATA

ANALYZING 95400 RECORDS


DATA QUALITY REPORT

Total records: 95400
Missing fields: 0 (0.0%)
  - Missing query: 0
  - Missing response: 0
Duplicate queries: 37594 (39.4%)

Type distribution:
  GSM_Rephrased       :  20028 ( 21.0%)
  GSM_AnsAug          :  18745 ( 19.6%)
  MATH_AnsAug         :  16999 ( 17.8%)
  MATH_Rephrased      :  12477 ( 13.1%)
  GSM_FOBAR           :  10023 ( 10.5%)
  GSM_SV              :   9869 ( 10.3%)
  MATH_FOBAR          :   3668 (  3.8%)
  MATH_SV             :   3591 (  3.8%)

Query length statistics:
  Min:          9 chars
  Q1 (25%):    132 chars
  Median:     198 chars
  Q3 (75%):    276 chars
  Max:       2310 chars
  Mean:       214 chars

Response length statistics:
  Min:         13 chars
  Q1 (25%):    132 chars
  Median:    

In [10]:
import os, sys, json, math, time, re, random, hashlib, inspect
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List
import re as _re
from collections import defaultdict, Counter
import math as _math

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
0 Tesla T4
1 Tesla T4


In [11]:
# ============================================================
# Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
))

TRAIN_FILE = "/kaggle/working/cleaned_data/train_cleaned.json"
VALID_FILE = DATA_DIR / "valid.json"

# PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải: "
PROMPT_TEMPLATE = (
    "Câu hỏi: {q}\n"
    "Hãy giải từng bước, mỗi bước là một phép tính.\n"
    "Kết thúc bằng '#### <số>'.\n"
    "Lời giải:\n"
) # try out
SAFE_EOS_ID = 50256

OUTPUT_DIR = Path("/kaggle/working/gpt2_math_ckpt")
VALID_OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
VALID_REPORT_PATH = Path("/kaggle/working/valid_report.json")
BASELINE_OUTPUT_PATH = Path("/kaggle/working/baseline_valid_output.json")
BASELINE_REPORT_PATH = Path("/kaggle/working/baseline_valid_report.json")

MAX_TRAIN_SAMPLES = 40000
MAX_VALID_SAMPLES = 1000

EPOCHS = 3
MAX_LENGTH = 768
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM = 8
LR = 3e-3
WARMUP_RATIO = 0.15
SEED = 42
WEIGHT_DECAY = 0.1

REPETITION_PENALTY = 1.0 # stop repeating words
NUM_BEAMS = 3 # Beam Search
LENGTH_PENALTY = 0.95 # smaller for concise answers?
NO_REPEAT_NGRAM_SIZE = 3

MAX_NEW_TOKENS = 256   # Đủ cho lời giải dài khi đã tách số
RUN_BASELINE_FIRST = False

VI_ANCHORS = [
    r"Câu trả lời là\s*[:：]?",
    r"Đáp án là\s*[:：]?",
    r"Đáp án\s*[:：]",
]

EN_ANCHORS = [
    r"The answer is\s*[:：]?",
    r"####",
]

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("TRAIN_FILE:", TRAIN_FILE)
print("VALID_FILE:", VALID_FILE)
print("MODEL_NAME:", MODEL_NAME)

TRAIN_FILE: /kaggle/working/cleaned_data/train_cleaned.json
VALID_FILE: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
MODEL_NAME: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


In [12]:
# Quick check: model must load offline
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)

tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print(type(tokenizer))
print(type(model))
print("vocab_size:", model.config.vocab_size)

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
vocab_size: 50257


In [13]:
# ============================================================
# 2. Cumulative Curriculum Learning
# ============================================================

def calculate_difficulty(record):
    """Đánh giá độ khó dựa trên số bước giải (số dấu '=') và loại bài"""
    response = record.get("response_vi", "")
    
    # Tiêu chí 1: Số bước giải (đếm số dấu bằng hoặc độ dài)
    steps = response.count("=") 
    length_penalty = len(response) / 500 # Phạt thêm nếu text quá dài
    
    # Tiêu chí 2: Loại bài
    type_score = 0
    record_type = record.get("type", "")
    if "MATH" in record_type:
        type_score += 2  # Toán cấp cao khó hơn
    if "FOBAR" in record_type or "SV" in record_type:
        type_score += 1  # Backward reasoning (tìm X) khó hơn giải xuôi
        
    return steps + length_penalty + type_score

In [14]:
# ============================================================
# Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)

# Tính điểm và sort data từ dễ đến khó
for rec in train_records:
    rec['difficulty'] = calculate_difficulty(rec)

# Sắp xếp lại list train_records từ dễ -> khó
train_records_sorted = sorted(train_records, key=lambda x: x['difficulty'])

# Phân chia thành 3 nhóm: Dễ (30%), Trung bình (35%), Khó (30%)
n = len(train_records_sorted)
easy_records = train_records_sorted[:int(0.3 * n)]
medium_records = train_records_sorted[int(0.3 * n):int(0.65 * n)]
hard_records = train_records_sorted[int(0.7 * n):]

# if MAX_TRAIN_SAMPLES:
#     train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train easy:", len(easy_records), "| train medium:", len(medium_records), "| train hard:", len(hard_records), "| valid:", len(valid_records))
print("sample medium query:", medium_records[0]["query_vi"][:200])

# ── Dataset statistics ──────────────────────────────────────
def _dataset_stats(records: list, label: str) -> None:
    """Print type distribution and response word-length percentiles."""
    types = Counter(r.get("type", "unknown") for r in records)
    print(f"\n[{label}] Type distribution ({len(types)} types):")
    for t, n in types.most_common():
        bar = "█" * min(40, n * 40 // max(types.values()))
        print(f"  {t:30s} {n:5d}  {bar}")
    lens = [len(r.get("response_vi", "").split()) for r in records]
    qs = [sorted(lens)[int(pct/100*(len(lens)-1))] for pct in [0,25,50,75,90,95,100]]
    print(f"[{label}] Response word-length p0/p25/p50/p75/p90/p95/p100: {qs}")

_dataset_stats(train_records, "TRAIN")
_dataset_stats(valid_records, "VALID")

train easy: 15955 | train medium: 18614 | train hard: 15956 | valid: 1000
sample medium query: Khi Jason chơi trò chơi điện tử Duty for Ashes, nhiệm vụ tiêu diệt kho rồng của anh ấy yêu cầu anh ấy phải bắn vũ khí của mình trung bình cứ sau x giây. Và mỗi lần anh ta bắn vũ khí, cài đặt hỏa lực m

[TRAIN] Type distribution (8 types):
  GSM_Rephrased                  16959  ████████████████████████████████████████
  MATH_Rephrased                  8145  ███████████████████
  GSM_SV                          8073  ███████████████████
  GSM_FOBAR                       7132  ████████████████
  GSM_AnsAug                      6222  ██████████████
  MATH_AnsAug                     3902  █████████
  MATH_SV                         1555  ███
  MATH_FOBAR                      1196  ██
[TRAIN] Response word-length p0/p25/p50/p75/p90/p95/p100: [5, 74, 105, 152, 193, 214, 341]

[VALID] Type distribution (8 types):
  GSM_AnsAug                       209  ████████████████████████████████████████
  GSM_

In [15]:
def force_answer_marker(text):
    if "####" in text:
        return text, True

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not nums:
        return text, False

    # take LAST number (usually final answer)
    return text + f" #### {nums[-1]}", True

In [16]:
# ============================================================
# 3. Hybrid Digit-Focus — Pre/Post-processing utilities
# ============================================================

_NUM_RE = _re.compile(r'-?\d+(?:[.,]\d+)*')

def _space_digits(m: _re.Match) -> str:
    """Space every character of a matched numeric token.

    "1500"  → "1 5 0 0"
    "1.5"   → "1 . 5"
    "1,5"   → "1 , 5"
    "-3.14" → "- 3 . 1 4"
    """
    return ' '.join(m.group())


def preprocess_text(text: str) -> str:
    """Expand every number in *text* to its digit-spaced form.

    Examples
    --------
    >>> preprocess_text("Lan có 1500 đồng và mua 250 kẹo.")
    'Lan có 1 5 0 0 đồng và mua 2 5 0 kẹo.'
    >>> preprocess_text("Câu trả lời là: 2000")
    'Câu trả lời là: 2 0 0 0'
    """
    return text


_COLLAPSE_RE = _re.compile(
    r'(?<!\S)'
    r'(-[ ])?'
    r'\d(?:[ ](?:[.,]|\d))*[ ]\d'
    r'(?!\S)'
)

def _collapse_match(m: _re.Match) -> str:
    return m.group().replace(' ', '')

def postprocess_text(text: str) -> str:
    """Collapse spaced-digit numbers back to compact form.

    Called on raw model output before extract_answer / parse_number.
    Safe to call multiple times (idempotent after first pass).
    """
    prev = None
    result = text
    while result != prev:
        prev = result
        result = _COLLAPSE_RE.sub(_collapse_match, result)
    return result

In [17]:
# ============================================================
# 4a. Enhanced Canonical Equation-Trace CoT Formatter
# ============================================================
# Applied to every extracted CoT step so the model trains on one
# consistent notation regardless of source format.
#
# Transformations (in order):
#   LaTeX delimiters  $…$ / $$…$$        stripped
#   \boxed{A}                             → A
#   \left / \right / \, / \! / …         stripped
#   \frac{A}{B}                           → (A)/(B)
#   \sqrt{A}                              → sqrt(A)
#   \pi / \theta / …                      → pi / theta / …
#   \cdot / \times / × / ·               → *
#   \div / ÷                              → /
#   **                                    → ^   (consistent power)
#   all whitespace                        removed  (compact form)
#   1,5  (Vietnamese decimal comma)       → 1.5
#   40% / x%                              → (40/100) / (x/100)
#   2x / 3( / )(                          → 2*x / 3*( / )*(

def normalize_equation(eq: str) -> str:
    """Normalize a math expression/equation to a compact consistent form."""
    if not eq:
        return eq

    # 1. Strip outer LaTeX display/inline delimiters
    eq = re.sub(r"^\$\$|\$\$$", "", eq.strip())
    eq = re.sub(r"^\\\[|\\\]$", "", eq.strip())
    eq = re.sub(r"^\$|\$$",     "", eq.strip())

    # 2. Unwrap \boxed{A} → A  (iterate to handle nesting)
    for _ in range(3):
        new = re.sub(r"\\boxed\{([^{}]*)\}", r"\1", eq)
        if new == eq:
            break
        eq = new

    # 3. Remove pure formatting commands (no math meaning)
    for tok in (r"\left", r"\right", r"\,", r"\!", r"\;",
                r"\ ", r"\quad", r"\qquad", r"\overline"
                r"\displaystyle", r"\textstyle"):
        eq = eq.replace(tok, "")
    eq = re.sub(r"\\text\{([^{}]*)\}",   r"\1", eq)
    eq = re.sub(r"\\mathrm\{([^{}]*)\}", r"\1", eq)

    # 4. \frac{A}{B} → (A)/(B)  [iterate for nested fractions]
    for _ in range(4):
        new = re.sub(
            r"\\(?:d|t)?frac\{([^{}]+)\}\{([^{}]+)\}",
            r"(\1)/(\2)",
            eq,
        )
        if new == eq:
            break
        eq = new

    # 5. \sqrt{A} → sqrt(A)
    eq = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", eq)
    eq = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", eq)

    # 6. Named constants and Greek letters
    eq = eq.replace("\\pi", "pi")
    for sym in ("alpha", "beta", "gamma", "theta", "lambda",
                "mu", "sigma", "phi", "omega", "Delta", "Sigma", "pmod"):
        eq = eq.replace(f"\\{sym}", sym)

    # 7. Multiplication operators → *
    eq = re.sub(r"\\cdot|\\times|[×·]", "*", eq)

    # 8. Division operators → /
    eq = re.sub(r"\\div|÷", "/", eq)

    # 9. Consistent power notation: ** → ^
    eq = eq.replace("**", "^")

    # 10. 
    eq = eq.replace("\\equiv", "≡")
    eq = eq.replace("\\le", "<=")

    # 11. Remove all whitespace — do this BEFORE implicit-mult detection
    #     so digit/letter boundaries are unambiguous
    eq = re.sub(r"\s+", "", eq)

    # 12. Vietnamese decimal comma: 1,5 → 1.5
    #     Matches 1–2 digits after comma (not thousands separator: 1,000)
    eq = re.sub(r"(\d),(\d{1,2})(?!\d)", r"\1.\2", eq)

    # 13. Percentage: 40% → (40/100),  x% → (x/100)
    eq = re.sub(r"(\d+(?:\.\d+)?)%", r"(\1/100)", eq)
    eq = re.sub(r"([A-Za-z])%",       r"(\1/100)", eq)

    # 14. Implicit multiplication (on compact form)
    #     digit before variable or opening paren:  2x → 2*x,  2( → 2*(
    eq = re.sub(r"(\d)([A-Za-z(])", r"\1*\2", eq)
    #     closing paren before opening paren or digit:  )( → )*(,  )2 → )*2
    eq = re.sub(r"\)([(\d])", r")*\1", eq)

    return eq.strip()

# Final answer anchor
ALL_ANCHORS = VI_ANCHORS + EN_ANCHORS

ANCHOR_RE = re.compile(
    rf"(?:{'|'.join(ALL_ANCHORS)})\s*\$?\s*(\\boxed\{{[^}}]+\}}|-?[\d.,/]+)",
    re.IGNORECASE
)

def clean_final_answer(ans: str) -> str:
    ans = ans.strip()

    # unwrap \boxed{...}
    ans = re.sub(r"\\boxed\{([^}]+)\}", r"\1", ans)

    # remove $ math delimiters
    ans = ans.replace("$", "")

    return ans.strip()

In [18]:
# ============================================================
# 4b. Enhanced Canonical Equation-Trace CoT Formatter
# ============================================================

# Split sentences
SENT_SPLIT_RE = re.compile(
    r'(?<=[.!?])\s+|\n+'
)

# Extract symbolic lines
SYMBOLIC_LINE_RE = re.compile(
    r"""
    (
        # anything containing '='
        [^.\n]*?=
        [^.\n]*

        |

        # latex-heavy expressions
        [^.\n]*
        (?:\\frac|\\sqrt|\\pi|\\sin|\\cos|\\tan|\\arcsin|\\arccos|\\arctan|\\boxed|\\cdot|\\times|\\div|\\binom|
    \\choose|\\sum|\\prod|\\log|\\ln|\\theta|\\alpha|\\beta|\\gamma)
        [^.\n]*

        |
        
        # exponents / powers
        [^.\n]*\^
        [^.\n]*

        |

        # factorials
        [^.\n]*!
        [^.\n]*

        |

        # percentage arithmetic
        [^.\n]*%
        [^.\n]*

        |

        # arithmetic chain
        [^.\n]*
        (?:
            \d+(?:[.,]\d+)?     # number
            |
            [a-zA-Z]            # variable
            |
            \\frac\{[^}]+\}\{[^}]+\}
        )
        [^.\n]*
        [+\-*/()]
        [^.\n]*
    )
    """,
    re.VERBOSE
)

SYMBOLIC_SPAN_RE = re.compile(
    r"""
    (
        # full equations
        [A-Za-z0-9\\{}()\[\].,:;<>|&+\-*/^%! ]+
        =
        [A-Za-z0-9\\{}()\[\].,:;<>|&+\-*/^%! ]+

        |

        # latex symbolic expressions
        \\(?:frac|sqrt|pi|sin|cos|tan|arcsin|arccos|arctan|
            boxed|cdot|times|div|binom|choose|
            sum|prod|log|ln|theta|alpha|beta|gamma)
        [A-Za-z0-9\\{}()\[\].,:;<>|&+\-*/^%! ]*
    )
    """,
    re.VERBOSE
)

# Must contain at least SOME mathematical structure
MATH_STRUCTURE_RE = re.compile(
    r"""
    =
    |
    [+\-*/^%]
    |
    \\[a-zA-Z]+
    |
    \d
    |
    [a-zA-Z]
    """,
    re.VERBOSE
)

def normalize_symbolic_line(s: str) -> str:
    """
    Clean symbolic lines while preserving math structure.
    """
    s = s.strip()

    # collapse whitespace
    s = re.sub(r"\s+", " ", s)

    # remove filler prefixes
    s = re.sub(
        r'^(Vì\ vậy|Do\ đó|Khi\ đó|Ta\ có|Chúng\ ta\ có|Suy\ ra|Giải\ ra|Rút\ gọn|Hãy\ thiết\ lập\ phương\ trình|viết|được|nhận\ được|ta\ được|chúng\ ta\ nhận\ được)\s*[:：]?\s*',
        '',
        s,
        flags=re.IGNORECASE
    )
    s = re.sub(r'^[A-Za-z\s:：.,;!?\-\+]+(?=\d|[a-zA-Z]|\(|\\)', '', s)
    s = s.lstrip(".,;:!? ：")
    # remove trailing punctuation
    s = s.rstrip(".,;:!?")
    s = re.sub(r'(?<=\d|\))[a-zA-Z\s]+$', '', s)

    return s.strip()

def enhanced_canonical_cot_formatter(text: str):
    """
    Convert natural-language math solution into compact equation-trace CoT.

    Supports:
    - multi-operand equations
    - chained equations
    - algebra
    - latex
    - fractions
    - percentages
    - factorials
    - geometry notation
    - decimals with , or .
    - variables
    """
    text = text.strip()

    # ── 1. Extract final answer ─────────────────────────────
    m = ANCHOR_RE.search(text)
    if not m:
        return text, False

    final_answer = clean_final_answer(m.group(1))

    # Remove answer section
    body = text[:m.start()].strip()

    # ── 2. Split into candidate spans ────────────────────────
    sentences = [
        s.strip()
        for s in SENT_SPLIT_RE.split(body)
        if s.strip()
    ]

    # ── 3. Extract symbolic transformation lines ────────────
    steps = []

    for sent in sentences:
        matches = SYMBOLIC_SPAN_RE.findall(sent)

        for span in matches:
            span = normalize_symbolic_line(span)

            # avoid tiny garbage
            if len(span) < 3:
                continue

            # must contain real math structure
            if not MATH_STRUCTURE_RE.search(span):
                continue

            # collapse spaces
            span = span.strip()

            if span and span not in steps:
                steps.append(span)

    # ── 4. Fallback if no equations found ───────────────────
    if not steps:
        fallback = []

        for s in sentences:
            if re.search(r'\d', s):
                s = normalize_symbolic_line(s)
                if s not in fallback:
                    fallback.append(s)

        # keep only first few
        steps = fallback[:4]

    # ── 5. Final fallback: never drop sample ────────────────
    if not steps:
        steps = [body]

    # ── 5. Build compact equation-trace CoT ────────────────
    lines = steps
    lines.append(f"#### {final_answer}")

    return "\n".join(lines), True

In [19]:
# ============================================================
# 5. Dynamic Prompting
# ============================================================
# 1. Ngân hàng ví dụ mẫu (1-shot) tương ứng với từng trường "type"
FEW_SHOT_BANK = {
    "GSM_Rephrased": {
        "query": "Nếu Alexis có 60 quả xoài và cô ấy có số xoài gấp bốn lần số xoài của Dilan và Ashley cộng lại thì tổng số xoài mà tất cả họ có được là bao nhiêu?",
        "response": "60 / 4 = 15\n60 + 15 = 75\n#### 75"
    },
    "MATH_Rephrased": {
        "query": "Xác định số nguyên dương nhỏ nhất chia hết cho ba số nguyên tố khác nhau.",
        "response": "2 * 3 * 5 = 30\n#### 30"
    },
    "GSM_AnsAug": {
        "query": "Angela cao hơn Helen 4 cm. Helen cao hơn Amy 3 cm. Nếu Amy cao 150 cm thì Angela cao bao nhiêu cm?",
        "response": "150 + 3 = 153\n153 + 4 = 157\n#### 157"
    },
    "MATH_AnsAug": {
        "query": "Nếu x - y = 6 và x + y = 12, giá trị của y là bao nhiêu?",
        "response": "x - y + x + y = 6 + 12\n2*x = 18\nx = 9\n9 + y = 12\ny = 3\n#### 3"
    },
    "GSM_FOBAR": {
        "query": "James đã kiếm được x đô la vào tháng 1. Tháng tiếp theo anh kiếm được gấp đôi số tiền đó. Tuy nhiên, vào tháng 3, James kiếm được ít hơn 2000 đô la so với tháng 2. James đã kiếm được bao nhiêu trong năm nay? Nếu chúng ta biết câu trả lời cho câu hỏi trên là 18000 thì giá trị của biến x chưa biết là bao nhiêu?",
        "response": "x + 2*x + 2*x - 2000 = 18000\n5*x - 2000 = 18000\n5*x = 20000\nx = 4000\n#### 4000"
    },
    "MATH_FOBAR": {
        "query": "Trừ X từ 333.33. Nếu chúng ta biết câu trả lời cho câu hỏi trên là 222.22 thì giá trị của biến X chưa biết là bao nhiêu?",
        "response": "333.33 - X = 222.22\n333.33 - 222.22 = X\nX = 111.11\n#### 111.11"
    },
    "GSM_SV": {
        "query": "Có 6 cô gái và x chàng trai tham gia vở kịch ở trường. Nếu cả bố và mẹ của mỗi đứa trẻ đều đến dự buổi ra mắt thì sẽ có 28 bố mẹ trong khán phòng. Giá trị của biến x chưa biết là bao nhiêu?",
        "response": "2 * (6 + x) = 28\n6 + x = 14\nx = 8\n#### 8"
    },
    "MATH_SV": {
        "query": "Một con súc sắc tám mặt có các mặt được đánh số từ 1 đến X. Giá trị kỳ vọng của con xúc xắc là 4.5. Giá trị của biến X chưa biết là bao nhiêu?",
        "response": "(1 + X) / 2 = 4.5\n1 + X = 9\nX = 8\n#### 8"
    }
}

# 2. Hàm định dạng prompt tích hợp Few-shot linh hoạt
def format_dataset_with_fewshot(raw_query: str, record_type: str, use_fewshot: bool = False) -> str:
    """
    Tạo cấu trúc Prompt tối ưu cho GPT-2.
    Nếu use_fewshot=True: Ép mô hình học theo ngữ cảnh 1 ví dụ mẫu chuẩn hóa.
    Nếu use_fewshot=False: Sử dụng Zero-shot với chỉ dẫn tối giản.
    """  
    # Định nghĩa chỉ dẫn ngắn gọn, không phân loại theo nhóm bài toán nữa
    instruction = "Tính kết quả theo từng bước phép tính toán học. Không dùng lời giải thích. Kết thúc bằng '#### <số>'.\n"
        
    # Xây dựng cấu trúc Prompt
    prompt_lines = []
    
    if use_fewshot and record_type in FEW_SHOT_BANK: # nhóm bài toán chỉ sử dụng khi định dùng few-shot, mình mặc định không dùng
        example = FEW_SHOT_BANK[record_type]
        # Chèn block ví dụ mẫu (1-Shot Context)
        prompt_lines.append(f"{instruction}")
        prompt_lines.append(f"Câu hỏi: {example['query']}")
        prompt_lines.append(f"Lời giải:\n{example['response']}\n---")
        
    # Chèn phần câu hỏi thực tế cần giải/huấn luyện
    if not use_fewshot:
        prompt_lines.append(f"{instruction}")
        
    prompt_lines.append(f"Câu hỏi: {raw_query}")
    prompt_lines.append("Lời giải:")
    
    return "\n".join(prompt_lines)

In [20]:
# ============================================================
# 6. SFT dataset. Stratified sampler and Length-aware collator
# ============================================================
# Stratified sampling by problem type
# ---------------------------------------------------------
# 1. Group all records by their "type" field.
# 2. Sample proportionally from each group (with a minimum floor
#    so rare types are not dropped entirely).
# 3. The result is a balanced subset that the model sees examples
#    from every problem category — critical for generalisation on
#    a held-out validation set that likely contains all types.
#
# LENGTH-BUCKETED BATCHING
# -------------------------
# Padding is the biggest GPU-time waster in sequence training.
# If a batch mixes a 50-token sample with a 480-token sample,
# the short sample wastes ~90% of its compute on padding tokens.
# Solution: sort training samples by (pre-tokenised) length, then
# shuffle within small windows.  This keeps batch lengths similar
# without making the order fully deterministic (which would cause
# the model to overfit to order artefacts).

def stratified_sample(
    records: list,
    max_samples: int,
    seed: int = SEED,
    min_per_type: int = 50,
) -> list:
    """Return up to *max_samples* records sampled proportionally by type.

    Every type gets at least *min_per_type* samples (or all of them if
    the type has fewer).  Remaining slots are filled proportionally.

    Parameters
    ----------
    records     : full list of training records
    max_samples : budget (e.g. MAX_TRAIN_SAMPLES)
    seed        : RNG seed for reproducibility
    min_per_type: guaranteed floor per type before proportional fill
    """
    rng = random.Random(seed)

    # Group by type
    groups: dict[str, list] = defaultdict(list)
    for r in records:
        groups[r.get("type", "unknown")].append(r)

    # Phase 1: guarantee floor for each type
    sampled: list = []
    remainder: dict[str, list] = {}
    for t, recs in groups.items():
        shuffled = recs[:]
        rng.shuffle(shuffled)
        take = min(min_per_type, len(shuffled))
        sampled.extend(shuffled[:take])
        remainder[t] = shuffled[take:]

    # Phase 2: fill remaining budget proportionally
    budget = max_samples - len(sampled)
    if budget > 0:
        total_rem = sum(len(v) for v in remainder.values())
        for t, recs in remainder.items():
            if not recs or total_rem == 0:
                continue
            quota = round(budget * len(recs) / total_rem)
            sampled.extend(recs[:quota])

    # Final shuffle and cap
    rng.shuffle(sampled)
    result = sampled[:max_samples]

    # Log the distribution of the final sample
    final_dist = Counter(r.get("type", "unknown") for r in result)
    print(f"[stratified_sample] {len(result)} records from {len(final_dist)} types")
    for t, n in sorted(final_dist.items(), key=lambda x: -x[1]):
        print(f"  {t:30s} {n:5d}")
    return result

class SFTDataset(Dataset):
    """Tokenize (prompt, response), mask loss on prompt + padding."""

    def __init__(self, records, tokenizer, max_length: int, is_training: bool):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length
        self.is_training = is_training
        # Pre-compute a length proxy (fast: character count as surrogate
        # for token count) used by the length-bucketed sampler.
        self._len_proxy = [
            len(r.get("query_vi", "")) + len(r.get("response_vi", ""))
            for r in records
        ]

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec = self.records[i]
        record_type = rec.get("type")
        if record_type is None:
            record_type = ""
        # Step 1 — enhanced equation trace CoT formatting (cell 2e)
        # Reformat the paragraph response into symbolic transformations.
        cot_response, _ = enhanced_canonical_cot_formatter(rec["response_vi"].strip())

        # Step 2 — Digit-Focus pre-processing (cell 2b)
        # Both query and CoT response are digit-spaced.
        raw_query    = preprocess_text(rec["query_vi"].strip())
        raw_response = preprocess_text(cot_response)
        
        #prompt   = PROMPT_TEMPLATE.format(q=raw_query)        
        prompt = format_dataset_with_fewshot(
            raw_query=raw_query, 
            record_type=record_type, 
            use_fewshot=False
        )
        
        response = raw_response

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        ids = (p_ids + r_ids)[: self.max_length]
        labels = ([-100] * len(p_ids) + r_ids)[: self.max_length]

        # Defensive clamp: never feed out-of-range ids to embedding layer.
        ids = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

def make_length_bucketed_indices(
    dataset: SFTDataset,
    bucket_size: int = 256,
    seed: int = SEED,
) -> list[int]:
    """Return index order that groups similar-length samples together.

    Algorithm:
      1. Sort all indices by character-length proxy.
      2. Split into buckets of *bucket_size* consecutive (similar-length)
         indices.
      3. Shuffle the order of buckets (so epochs don't always start with
         short samples), then shuffle within each bucket.

    This keeps padding waste low while preserving enough randomness that
    the model does not overfit to length ordering.

    Usage: pass the returned list to a SubsetRandomSampler or use it
    directly as the dataset order via a custom Sampler.
    """
    rng = random.Random(seed)
    # Sort by length proxy
    sorted_idx = sorted(range(len(dataset)), key=lambda i: dataset._len_proxy[i])
    # Split into buckets
    buckets = [sorted_idx[i:i+bucket_size] for i in range(0, len(sorted_idx), bucket_size)]
    # Shuffle bucket order, then shuffle within each bucket
    rng.shuffle(buckets)
    for b in buckets:
        rng.shuffle(b)
    # Flatten
    return [idx for bucket in buckets for idx in bucket]

class LengthBucketedSampler(torch.utils.data.Sampler):
    """Sampler that yields indices in length-bucketed order.

    Pass this to Trainer via a subclassed Trainer that overrides
    _get_train_dataloader, or use it directly with a DataLoader.
    Since HuggingFace Trainer does not natively accept a custom sampler
    we subclass Trainer below to inject it.
    """
    def __init__(self, dataset: SFTDataset, bucket_size: int = 256, seed: int = SEED):
        self.dataset = dataset
        self.bucket_size = bucket_size
        self.seed = seed
        self._indices = make_length_bucketed_indices(dataset, bucket_size, seed)

    def __iter__(self):
        return iter(self._indices)

    def __len__(self):
        return len(self._indices)


class BucketedTrainer(Trainer):
    """Trainer subclass that uses LengthBucketedSampler for training.
    Everything else (eval, saving, logging) is unchanged.
    """
    def _get_train_dataloader(self):
        sampler = LengthBucketedSampler(
            self.train_dataset,
            bucket_size=256,
            seed=self.args.seed,
        )
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=True,
        )

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

In [21]:
# ============================================================
# Evaluation utilities
# ============================================================
BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}


def extract_answer(text: str | None, anchors: list[str]) -> str | None:
    """Extract final answer by anchors. First anchor wins; fallback to last \\boxed{}."""
    if not text:
        return None

    for anc in anchors:
        m = re.search(anc, text)
        if m:
            tail = text[m.end():].strip()
            # find first numeric-looking expression
            m2 = re.search(r"-?[\d.,/]+", tail)

            if m2:
                return m2.group(0)

    boxes = BOXED_RE.findall(text)
    if boxes:
        return boxes[-1].strip()

    return None


def extract_gold(rec: dict) -> str | None:
    """Gold answer is read only from gold records, not from prediction files."""
    return extract_answer(rec.get("response_vi"), VI_ANCHORS + EN_ANCHORS)


def extract_pred(rec: dict) -> str | None:
    """Predicted answer is read only from model_output."""
    return extract_answer(rec.get("model_output"), VI_ANCHORS + EN_ANCHORS)


def parse_number(s: str | None) -> float | None:
    """Best-effort parser for finite scalar numeric answers."""
    if s is None:
        return None

    t = s.strip()
    if not t:
        return None

    # Pure Vietnamese decimal: 1,5 -> 1.5
    if re.fullmatch(r"-?\d+,\d+", t):
        try:
            return float(t.replace(",", "."))
        except ValueError:
            return None

    # Pure English decimal / integer
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", t):
        try:
            val = float(t)
            return val if math.isfinite(val) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")

    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

    # Implicit multiplication
    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")

    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)

    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    t = t.replace("^", "**")

    try:
        val = eval(t, {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(val, bool):
        return None

    if isinstance(val, (int, float)):
        val = float(val)
        return val if math.isfinite(val) else None

    return None


def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None

    denom = max(1.0, abs(gold))
    return abs(pred - gold) / denom


def score_one(re_val: float | None, extractable: bool) -> int:
    if not extractable:
        return 0
    if re_val is None:
        return 0
    if re_val <= 0.01:
        return 10
    if re_val <= 0.10:
        return 5
    if re_val <= 0.50:
        return 1
    return 0


def align_predictions_with_gold(pred_items: list[dict], gold_items: list[dict]) -> list[tuple[dict, dict]]:
    """
    Align predictions and gold records.

    If both sides have id, align by id.
    Otherwise, require the same length and align by order.
    """
    pred_has_id = all("id" in x for x in pred_items)
    gold_has_id = all("id" in x for x in gold_items)

    if pred_has_id and gold_has_id:
        pred_map = {str(x["id"]): x for x in pred_items}
        pairs = []

        missing = []
        for g in gold_items:
            gid = str(g["id"])
            if gid not in pred_map:
                missing.append(gid)
            else:
                pairs.append((pred_map[gid], g))

        if missing:
            raise ValueError(f"Prediction thiếu {len(missing)} id, ví dụ: {missing[:5]}")

        return pairs

    if len(pred_items) != len(gold_items):
        raise ValueError(
            f"Số lượng prediction ({len(pred_items)}) khác số lượng gold ({len(gold_items)})."
        )

    return list(zip(pred_items, gold_items))


def evaluate(pred_items: list[dict], gold_items: list[dict]) -> dict:
    pairs = align_predictions_with_gold(pred_items, gold_items)

    rows = []
    total = 0
    bucket10 = bucket5 = bucket1 = bucket0 = 0
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in pairs:
        gold_ans = extract_gold(gold_rec)
        pred_ans = extract_pred(pred_rec)

        is_extractable = pred_ans is not None
        extractable += int(is_extractable)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)
        re_val = rel_error(pred_num, gold_num)

        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s

        bucket10 += int(s == 10)
        bucket5 += int(s == 5)
        bucket1 += int(s == 1)
        bucket0 += int(s == 0)

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    max_score = n * 10
    score_10 = total / n if n else 0.0

    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": max_score,
            "score_10": score_10,
            "score_pct": total / max_score if max_score else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": {
                "10": bucket10,
                "5": bucket5,
                "1": bucket1,
                "0": bucket0,
            },
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }


def save_evaluation_report(
    pred_path: str | Path,
    gold_records: list[dict],
    report_path: str | Path,
) -> dict:
    pred_path = Path(pred_path)
    report_path = Path(report_path)

    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)

    result = evaluate(pred_items, gold_records)

    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Wrote {report_path}")
    return result

In [22]:
# ============================================================
# 7. Decoding Strategy - Self-Consistency
# ============================================================

from collections import Counter

def majority_vote(answers):
    answers = [a for a in answers if a is not None]

    if not answers:
        return None, 0

    counter = Counter(answers)
    most_common_ans, vote_count = counter.most_common(1)[0]
    return most_common_ans, vote_count
    
def self_consistency_generate(
    model,
    tokenizer,
    prompt,
    vocab_n,
    max_new_tokens,
    num_samples=20,
    temperature=0.5,
):

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    ids = enc["input_ids"].clamp(max=vocab_n - 1)

    gen = model.generate(
        input_ids=ids,
        attention_mask=enc.get("attention_mask"),

        max_new_tokens=max_new_tokens,

        # sampling
        do_sample=True,
        temperature=temperature,
        top_p=0.85,

        # IMPORTANT
        num_return_sequences=num_samples,

        # self-consistency nên beam = 1
        num_beams=1,

        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,

        pad_token_id=SAFE_EOS_ID,
        eos_token_id=SAFE_EOS_ID,
    )

    paths = []
    answers = []

    for i in range(num_samples):

        raw_text = tokenizer.decode(
            gen[i, ids.shape[1]:],
            skip_special_tokens=True
        )

        text = postprocess_text(raw_text)

        paths.append(text)

        answer = extract_answer(
            text,
            ALL_ANCHORS
        )

        answers.append(answer)

    final_answer, vote_count = majority_vote(answers)

    if final_answer is not None:
        best_idx = answers.index(final_answer)
        final_path = paths[best_idx]
    else:
        # Nếu tất cả các mẫu đều không rút trích được số, lấy mẫu đầu tiên làm fallback
        final_path = paths[0] if paths else ""

    return {
        "paths": paths,
        "answers": answers,
        "final_answer": final_answer,
        "final_path": final_path, # Giữ lại lời giải đầy đủ của chuỗi nhất quán nhất
    }

In [23]:
# ============================================================
# Output Generation
# ============================================================

from collections import Counter

def majority_vote(answers):
    answers = [a for a in answers if a is not None]

    if not answers:
        return None, 0

    counter = Counter(answers)
    most_common_ans, vote_count = counter.most_common(1)[0]
    return most_common_ans, vote_count
    
def build_prompt(rec: dict) -> str:
    # Apply same digit-spacing as training so the model sees the same
    # token distribution it was fine-tuned on.
    spaced_q = preprocess_text(rec["query_vi"].strip())
    return PROMPT_TEMPLATE.format(q=spaced_q)

def generate_outputs(model_path_or_name: str | Path, records: list, output_path: str | Path, max_new_tokens: int = 256):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading {model_path_or_name} on {device} ...", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(model_path_or_name, local_files_only=True)
    tokenizer.pad_token_id = SAFE_EOS_ID
    tokenizer.eos_token_id = SAFE_EOS_ID

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path_or_name,
        torch_dtype=dtype,
        local_files_only=True,
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    # ── GPU memory snapshot ──────────────────────────────────────
    if device == "cuda":
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        print(f"[inference] GPU memory: {allocated:.2f} GB allocated, "
              f"{reserved:.2f} GB reserved", flush=True)

    vocab_n = model.transformer.wte.num_embeddings
    outputs, t0 = [], time.time()
    n_total   = len(records)
    log_every = max(1, n_total // 10)          # log ~10 progress lines
    n_empty   = 0                              # tracks outputs with no answer anchor

    with torch.inference_mode():
        for rec in tqdm(records, desc="Generating"):
            # build_prompt() applies preprocess_text() internally (cell 2b)
            # — no need to duplicate it here.
            # prompt = build_prompt(rec)
            raw_query    = preprocess_text(rec["query_vi"].strip())
            record_type=rec.get("type")
            if record_type is None:
                record_type = ""
            
            prompt = format_dataset_with_fewshot(
                raw_query=raw_query, 
                record_type=record_type,
                use_fewshot=False
            )
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

            ids = enc["input_ids"].clamp(max=vocab_n - 1)
            result = self_consistency_generate(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                vocab_n=vocab_n,
                max_new_tokens=max_new_tokens,
            
                # self-consistency params
                num_samples=20,
                temperature=0.5,
            )
            
            chosen_full_text = result["final_path"]
            
            if result["final_answer"] == 0:
                n_empty += 1
            
            outputs.append({
                "id": len(outputs),
                "query_vi": rec["query_vi"],
                "type": rec.get("type"),
            
                # final voted answer
                "model_output": chosen_full_text,
            
                # # optional debug
                # "voted_answer": result["final_answer"],
                # "all_answers": result["answers"],
                # "all_paths": result["paths"],
            })

            

            # ── Periodic progress log ────────────────────────────────
            done = len(outputs)
            if done % log_every == 0 or done == n_total:
                elapsed   = time.time() - t0
                rate      = done / elapsed if elapsed > 0 else 0         # samples/sec
                remaining = (n_total - done) / rate if rate > 0 else 0
                print(
                    f"[inference] {done}/{n_total}  "
                    f"elapsed={elapsed/60:.1f}min  "
                    f"eta={remaining/60:.1f}min  "
                    f"no-anchor={n_empty} ({100*n_empty/done:.1f}%)",
                    flush=True,
                )

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)

    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    return outputs

In [24]:
# ============================================================
# Optional baseline
# ============================================================
if RUN_BASELINE_FIRST:
    _ = generate_outputs(MODEL_NAME, valid_records, BASELINE_OUTPUT_PATH, MAX_NEW_TOKENS)
    baseline_result = save_evaluation_report(BASELINE_OUTPUT_PATH, valid_records, BASELINE_REPORT_PATH)
else:
    print("Skip baseline.")

Skip baseline.


In [25]:
collator  = PadCollator(pad_id=SAFE_EOS_ID)
eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())

class EpochTimingCallback(TrainerCallback):
    """Logs wall time and estimated remaining time after each epoch."""
    def __init__(self, total_epochs):
        self._t_start = None
        self._epoch_times = []
        self.total_epochs = total_epochs

    def on_train_begin(self, args, state, control, **kwargs):
        self._t_start = time.time()
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1e9
            res   = torch.cuda.memory_reserved()  / 1e9
            print(f"[train] GPU memory at start: "
                  f"{alloc:.2f} GB allocated, {res:.2f} GB reserved", flush=True)

    def on_epoch_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self._t_start
        epoch   = int(state.epoch)
        self._epoch_times.append(elapsed)
        avg_ep  = elapsed / epoch if epoch else elapsed
        remaining = avg_ep * (self.total_epochs - epoch)
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1e9
        else:
            alloc = 0.0
        print(
            f"[train] Epoch {epoch} done  "
            f"total_elapsed={elapsed/60:.1f}min  "
            f"avg_per_epoch={avg_ep/60:.1f}min  "
            f"eta={remaining/60:.1f}min  "
            f"gpu_alloc={alloc:.2f}GB",
            flush=True,
        )
        # Warn if we are close to the 3-hour budget
        budget_sec = 3 * 3600
        if elapsed + remaining > budget_sec * 0.9:
            print(f"  ⚠  Projected total ({(elapsed+remaining)/60:.0f}min) "
                  f"may exceed 3-hour budget!", flush=True)

# ── Stratified sampling (see cell 3 for rationale) ──────────
sampled_easy_train = stratified_sample(easy_records, len(easy_records), seed=SEED)
sampled_medium_train = stratified_sample(medium_records, len(medium_records), seed=SEED)
sampled_hard_train = stratified_sample(hard_records, len(hard_records), seed=SEED)

cleaned_easy_train = []
for r in sampled_easy_train:
    text, ok = force_answer_marker(r["response_vi"])
    if not ok:
        continue
    r["response_vi"] = text
    cleaned_easy_train.append(r)

sampled_easy_train = cleaned_easy_train

cleaned_medium_train = []
for r in sampled_medium_train:
    text, ok = force_answer_marker(r["response_vi"])
    if not ok:
        continue
    r["response_vi"] = text
    cleaned_medium_train.append(r)

sampled_medium_train = cleaned_medium_train

cleaned_hard_train = []
for r in sampled_hard_train:
    text, ok = force_answer_marker(r["response_vi"])
    if not ok:
        continue
    r["response_vi"] = text
    cleaned_hard_train.append(r)

sampled_hard_train = cleaned_hard_train

# ============================================================
# Định nghĩa các Giai đoạn Cumulative (Cumulative Stages)
# ============================================================
stages_config = [
    {"name": "STAGE 1: WARM-UP (EASY ONLY)", "records": sampled_easy_train, "epochs": 2, "learning_rate": 4e-3, "warmup_ratio": 0.15},
    {"name": "STAGE 2: CUMULATIVE (EASY + MEDIUM)", "records": sampled_easy_train + sampled_medium_train, "epochs": 2, "learning_rate": 3e-3, "warmup_ratio": 0.10},
    {"name": "STAGE 3: FULL DISTRIBUTION (ALL)", "records": sampled_easy_train + sampled_medium_train + sampled_hard_train, "epochs": 2, "learning_rate": 2e-3, "warmup_ratio": 0.05}
]

TOTAL_CUMULATIVE_EPOCHS = sum(s["epochs"] for s in stages_config)
timing_callback = EpochTimingCallback(total_epochs=TOTAL_CUMULATIVE_EPOCHS)

[stratified_sample] 15955 records from 8 types
  GSM_Rephrased                   9554
  GSM_AnsAug                      3484
  MATH_Rephrased                  1763
  MATH_AnsAug                      782
  GSM_SV                           123
  GSM_FOBAR                        111
  MATH_SV                          103
  MATH_FOBAR                        35
[stratified_sample] 18614 records from 8 types
  GSM_Rephrased                   6320
  GSM_FOBAR                       3946
  MATH_Rephrased                  2904
  GSM_AnsAug                      2332
  MATH_AnsAug                     1379
  GSM_SV                          1263
  MATH_SV                          265
  MATH_FOBAR                       205
[stratified_sample] 15956 records from 8 types
  GSM_SV                          6021
  MATH_Rephrased                  2890
  GSM_FOBAR                       2435
  MATH_AnsAug                     1490
  MATH_SV                         1134
  MATH_FOBAR                       887
 

In [26]:
# ============================================================
# Train
# ============================================================
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

model.gradient_checkpointing_enable()
model.config.use_cache = False

valid_ds = SFTDataset(valid_records,  tokenizer, MAX_LENGTH, is_training=False)

for i, stage in enumerate(stages_config):
    print(f"\n" + "="*60)
    print(f"🚀 {stage['name']}")
    print(f"   Số lượng samples: {len(stage['records'])}")
    print(f"="*60)
    
    # Tạo bản sao và shuffle danh sách record của stage này để xáo trộn độ khó dễ
    stage_records = list(stage["records"])
    random.shuffle(stage_records)

    # BẮT BUỘC: Khởi tạo lại Class Dataset của bạn với tập dữ liệu mới
    stage_train_ds = SFTDataset(stage_records, tokenizer, max_length=MAX_LENGTH, is_training=True)
    
    steps_per_epoch = math.ceil(len(stage_train_ds) / eff_batch)
    print(
        f"per_device_bs={PER_DEVICE_BATCH_SIZE} | grad_accum={GRAD_ACCUM} | "
        f"gpus={torch.cuda.device_count()} | effective_batch={eff_batch} | "
        f"steps/epoch={steps_per_epoch}"
    )

    ta_kwargs = dict(
        output_dir=str(OUTPUT_DIR) + f"_stage_{i+1}",
        num_train_epochs=stage["epochs"],
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=stage["learning_rate"],
        optim="adamw_torch",
        warmup_ratio=stage["warmup_ratio"],
        lr_scheduler_type="cosine_with_restarts",
        weight_decay=WEIGHT_DECAY,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        save_strategy="no",
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )

    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        ta_kwargs["eval_strategy"] = "epoch"
    else:
        ta_kwargs["evaluation_strategy"] = "epoch"

    training_args = TrainingArguments(**ta_kwargs)

    # BucketedTrainer uses LengthBucketedSampler to reduce padding waste.
    trainer = BucketedTrainer(
        model=model,
        args=training_args,
        train_dataset=stage_train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        callbacks=[timing_callback],
    )

    trainer.train()
    # Giải phóng ram của riêng Trainer hiện tại, giữ lại `model` cho stage sau
    del trainer
    torch.cuda.empty_cache()

# ============================================================
# 4. Lưu Checkpoint cuối cùng sau khi hoàn thành tất cả Stage
# ============================================================
train_dt = time.time() - t0
print(f"\n[🏆 ALL STAGES COMPLETE] Total wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model_hash = sha256_dir(OUTPUT_DIR)
with (OUTPUT_DIR / "model_hash.txt").open("w", encoding="utf-8") as f:
    f.write(model_hash + "\n")

print("Saved checkpoint to:", OUTPUT_DIR)
print("Model SHA256:", model_hash)

# Free training objects before inference reload.
del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 STAGE 1: WARM-UP (EASY ONLY)
   Số lượng samples: 15953
per_device_bs=4 | grad_accum=8 | gpus=2 | effective_batch=64 | steps/epoch=250


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[train] GPU memory at start: 0.51 GB allocated, 0.57 GB reserved


...


🚀 STAGE 3: FULL DISTRIBUTION (ALL)
   Số lượng samples: 50517
per_device_bs=4 | grad_accum=8 | gpus=2 | effective_batch=64 | steps/epoch=790
[train] GPU memory at start: 0.54 GB allocated, 0.81 GB reserved


Epoch,Training Loss,Validation Loss
1,0.789778,1.095623
2,0.467992,1.094332


[train] Epoch 1 done  total_elapsed=42.5min  avg_per_epoch=42.5min  eta=212.5min  gpu_alloc=1.54GB
  ⚠  Projected total (255min) may exceed 3-hour budget!
[train] Epoch 2 done  total_elapsed=85.3min  avg_per_epoch=42.7min  eta=170.6min  gpu_alloc=1.54GB
  ⚠  Projected total (256min) may exceed 3-hour budget!

[🏆 ALL STAGES COMPLETE] Total wall time: 9478.0s (157.97 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to: /kaggle/working/gpt2_math_ckpt
Model SHA256: 5e2d3aa5c9aeba2a4041aa1f01adad09ed647979ec57fbddd89b758b6b353351


In [27]:
# ============================================================
# Inference on validation set
# ============================================================
# OUTPUT_DIR_2 = Path("/kaggle/input/datasets/trangdngminh/fine-tune-gpt-2-planning-lm/gpt2_math_ckpt")
valid_outputs = generate_outputs(OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS)

print("\nExample output:")
print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

Loading /kaggle/working/gpt2_math_ckpt on cuda ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[inference] GPU memory: 1.80 GB allocated, 1.98 GB reserved


Generating:   0%|          | 0/1000 [00:00<?, ?it/s]

[inference] 100/1000  elapsed=1.9min  eta=16.8min  no-anchor=0 (0.0%)
[inference] 200/1000  elapsed=3.9min  eta=15.7min  no-anchor=0 (0.0%)
[inference] 300/1000  elapsed=5.8min  eta=13.6min  no-anchor=0 (0.0%)
[inference] 400/1000  elapsed=7.3min  eta=10.9min  no-anchor=0 (0.0%)
[inference] 500/1000  elapsed=9.1min  eta=9.1min  no-anchor=0 (0.0%)
[inference] 600/1000  elapsed=11.0min  eta=7.3min  no-anchor=0 (0.0%)
[inference] 700/1000  elapsed=12.8min  eta=5.5min  no-anchor=0 (0.0%)
[inference] 800/1000  elapsed=14.3min  eta=3.6min  no-anchor=0 (0.0%)
[inference] 900/1000  elapsed=16.0min  eta=1.8min  no-anchor=0 (0.0%)
[inference] 1000/1000  elapsed=18.1min  eta=0.0min  no-anchor=0 (0.0%)
Wrote /kaggle/working/valid_output.json | 18.08 min | SHA256: 4d4bd9486703d826fc097151bc733a3da1b120ccb26a2b676d62ad72bf0ef138

Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước

In [28]:
# ============================================================
# Evaluate
# ============================================================
valid_result = save_evaluation_report(
    VALID_OUTPUT_PATH,
    valid_records,
    VALID_REPORT_PATH,
)

summary = valid_result["summary"]

print("\nFinal validation score:")
print(f'{summary["raw_score"]} / {summary["max_raw_score"]}  ({summary["score_pct"]*100:.2f}%)')
print(f'Score /10: {summary["score_10"]:.2f}')
print("Buckets:", summary["buckets"])

{
  "n": 1000,
  "raw_score": 3663,
  "max_raw_score": 10000,
  "score_10": 3.663,
  "score_pct": 0.3663,
  "extractable": 1000,
  "numeric_pairs": 992,
  "buckets": {
    "10": 340,
    "5": 18,
    "1": 173,
    "0": 469
  },
  "rel_error_mean": 12.575136332736827
}
Wrote /kaggle/working/valid_report.json

Final validation score:
3663 / 10000  (36.63%)
Score /10: 3.66
Buckets: {'10': 340, '5': 18, '1': 173, '0': 469}


In [29]:
# ============================================================
# Error analysis preview
# ============================================================
def analyze_accuracy_by_type(rows):
    """
    Phân tích tỷ lệ chính xác (accuracy) và điểm trung bình cho từng dạng bài toán.
    Args:
        rows (list): Danh sách các dict kết quả, ví dụ: valid_result["rows"]
    
    Returns:
        dict: Chứa thống kê chi tiết cho từng dạng bài.
    """
    # Khởi tạo dictionary để lưu trữ số liệu
    stats = defaultdict(lambda: {"total": 0, "correct": 0, "total_score": 0})
    
    for row in rows:
        q_type = row.get("type", "Unknown")
        score = row.get("score", 0)
        
        stats[q_type]["total"] += 1
        stats[q_type]["total_score"] += score
        
        # Điểm 10 tương đương với việc mô hình trả lời hoàn toàn chính xác
        if score == 10:
            stats[q_type]["correct"] += 1
            
    # In bảng báo cáo
    print(f"{'Dạng bài (Type)':<20} | {'Tổng số':<10} | {'Số câu đúng':<15} | {'Accuracy (%)':<15} | {'Avg Score (/10)'}")
    print("-" * 80)
    
    # Sắp xếp hiển thị theo số lượng câu hỏi giảm dần
    sorted_stats = sorted(stats.items(), key=lambda x: x[1]['total'], reverse=True)
    for q_type, data in sorted_stats:
        total = data["total"]
        correct = data["correct"]
        acc = (correct / total) * 100 if total > 0 else 0
        avg_score = data["total_score"] / total if total > 0 else 0
        
        print(f"{q_type:<20} | {total:<10} | {correct:<15} | {acc:<15.2f} | {avg_score:.2f}")
        
    return dict(stats)


rows = valid_result["rows"]
bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
good = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 10]

print("Good:", len(good), "| Bad:", len(bad))

for i, r in bad[:2]:
    pred = valid_outputs[i]
    gold = valid_records[i]
    print("=" * 100)
    print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
    print("QUERY:", gold["query_vi"][:500])
    print("\nGOLD:", gold["response_vi"][-500:])
    print("\nPRED:", pred["model_output"][:1000])

type_statistics = analyze_accuracy_by_type(rows=rows)

Good: 340 | Bad: 469
IDX: 1 | type: MATH_Rephrased | rel_error: 0.7368421052631579
QUERY: Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?

GOLD: Vì $\angle PQR = \angle PRQ$ nên ta biết tam giác PQR là tam giác cân. Điều này có nghĩa là PQ cũng có độ dài bằng 7. Khi đó chu vi của tam giác PQR là $PQ + QR + PR = 7 + 5 + 7 = \boxed{19}$.Câu trả lời là: 19

PRED: \angle PQr = \frac{1}{2} \angle PQQ
\angle RQ = \boxed{5}
#### 5
IDX: 2 | type: MATH_SV | rel_error: 3.5
QUERY: Một con súc sắc tám mặt có các mặt được đánh số từ 1 đến X. Giá trị kỳ vọng của con xúc xắc là 4,5. Giá trị của biến X chưa biết là bao nhiêu?

GOLD: Để giải bài toán này, chúng ta cần xác định giá trị của x, biểu thị số ở mặt cao nhất của xúc xắc tám mặt. Giá trị kỳ vọng của việc tung xúc xắc được cho là 4,5. Điều này có nghĩa là trung bình, tổng các số trên tất cả các mặt chia cho số mặt là 4,5. Hãy thiết lập phương trình: (1 + 2 + 3 + 4 + 5 + 6 + 7 + 

In [30]:
# ============================================================
# List saved artifacts
# ============================================================
for p in [
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    OUTPUT_DIR / "model_hash.txt",
]:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)

print("\nKaggle output folder:")
for p in sorted(Path("/kaggle/working").glob("*")):
    print("-", p)

/kaggle/working/valid_output.json exists= True size= 453080
/kaggle/working/valid_report.json exists= True size= 233292
/kaggle/working/valid_output.json.sha256.txt exists= True size= 65
/kaggle/working/gpt2_math_ckpt/model_hash.txt exists= True size= 65

Kaggle output folder:
- /kaggle/working/__notebook__.ipynb
- /kaggle/working/cleaned_data
- /kaggle/working/gpt2_math_ckpt
- /kaggle/working/gpt2_math_ckpt_stage_1
- /kaggle/working/gpt2_math_ckpt_stage_2
- /kaggle/working/gpt2_math_ckpt_stage_3
- /kaggle/working/valid_output.json
- /kaggle/working/valid_output.json.sha256.txt
- /kaggle/working/valid_report.json
